In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_community.llms import Tongyi

# 获取环境中KEY
tongyi_key = os.environ.get('QWEN_KEY')
# 设置KEY
os.environ["DASHSCOPE_API_KEY"] = tongyi_key
# 创建模型
llm = Tongyi()

In [2]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
# 获取格式的提示词
format_inst = parser.get_format_instructions()
# 提示词
prompt_text = f"""
请分析以下产品评价，并返回一个JSON对象。
要求：包含product_name(产品名)、rating(评分1-5)、pros(优点列表)、cons(缺点列表)

格式要求：
{format_inst}

评价内容：这款手机电池续航很好，能使用一整天，但摄像头效果一般，价格偏高。
"""

# 获取模型输出并解析
raw_output = llm.invoke(prompt_text)
rs = parser.parse(raw_output)

print('解析后的JSON数据：')
print(rs)
print(f'产品名: {rs.get("product_name")}')

解析后的JSON数据：
{'product_name': '手机', 'rating': 3, 'pros': ['电池续航很好', '能使用一整天'], 'cons': ['摄像头效果一般', '价格偏高']}
产品名: 手机


In [8]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field
from typing import List

# 定义结构
class ArticleSummary(BaseModel):
    title: str = Field(description="文章标题")
    author: str = Field(description="作者")
    key_points: List[str] = Field(description="关键点列表", min_length=3)
    summary: str = Field(description="文章摘要", min_length=50)

# 初始化
parser = PydanticOutputParser(pydantic_object=ArticleSummary)

# 获取类型提示词
instructions = parser.get_format_instructions()

user_query = "请总结《红楼梦》前五回的主要内容"

prompt = f"""
你是一个文学分析助手。
请根据用户请求分析文章内容。

输出格式必须严格遵守：
{instructions}

用户请求：{user_query}
"""

response = llm.invoke(prompt)
response


'{\n  "title": "《红楼梦》前五回主要内容总结",\n  "author": "曹雪芹",\n  "key_points": [\n    "以女娲补天遗石和神瑛侍者、绛珠仙草的神话为引，构建全书‘真事隐去，假语村言’的叙事框架与‘木石前盟’的情感宿命。",\n    "通过甄士隐的盛衰际遇（元宵失女、葫芦庙失火、家道败落）与贾雨村的仕途起落（受助赴考、中举为官、徇私枉法），形成‘真’‘假’对照，揭示世态炎凉与命运无常。",\n    "第五回贾宝玉梦游太虚幻境，阅‘金陵十二钗’判词及《红楼梦》十二支曲，预示主要女性角色的命运轨迹与贾府‘好一似食尽鸟投林，落了片白茫茫大地真干净’的总体悲剧结局。"\n  ],\n  "summary": "《红楼梦》前五回构成全书的总纲与序曲。第一回以神话开篇，交代青埂峰下顽石入世、神瑛绛珠‘木石前盟’的渊源，并点明‘真事隐’‘假语存’的创作手法；第二至四回通过甄士隐的悲剧性退场与贾雨村的功利性登场，完成由‘真’向‘假’、由隐逸向世俗的过渡，同时引出贾府核心人物及社会关系网络；第五回是全书关键枢纽，宝玉梦游太虚幻境，所见判词、曲文与‘情榜’暗藏众女子命运伏线，将神话寓言、现实叙事与哲学观照熔铸一体，奠定全书‘千红一哭，万艳同悲’的悲剧基调与宿命结构。这五回不仅铺陈背景、勾勒人物、确立风格，更以高度凝练的象征系统，为后续百回巨构埋下严密而深邃的伏笔。"\n}'

In [10]:
rs = parser.parse(response)
print(f'标题:{rs.title}')
print(f'作者:{rs.author}')
print(f'关键点列表:{rs.key_points}')
print(f'文章摘要:{rs.summary}')

标题:《红楼梦》前五回主要内容总结
作者:曹雪芹
关键点列表:['以女娲补天遗石和神瑛侍者、绛珠仙草的神话为引，构建全书‘真事隐去，假语村言’的叙事框架与‘木石前盟’的情感宿命。', '通过甄士隐的盛衰际遇（元宵失女、葫芦庙失火、家道败落）与贾雨村的仕途起落（受助赴考、中举为官、徇私枉法），形成‘真’‘假’对照，揭示世态炎凉与命运无常。', '第五回贾宝玉梦游太虚幻境，阅‘金陵十二钗’判词及《红楼梦》十二支曲，预示主要女性角色的命运轨迹与贾府‘好一似食尽鸟投林，落了片白茫茫大地真干净’的总体悲剧结局。']
文章摘要:《红楼梦》前五回构成全书的总纲与序曲。第一回以神话开篇，交代青埂峰下顽石入世、神瑛绛珠‘木石前盟’的渊源，并点明‘真事隐’‘假语存’的创作手法；第二至四回通过甄士隐的悲剧性退场与贾雨村的功利性登场，完成由‘真’向‘假’、由隐逸向世俗的过渡，同时引出贾府核心人物及社会关系网络；第五回是全书关键枢纽，宝玉梦游太虚幻境，所见判词、曲文与‘情榜’暗藏众女子命运伏线，将神话寓言、现实叙事与哲学观照熔铸一体，奠定全书‘千红一哭，万艳同悲’的悲剧基调与宿命结构。这五回不仅铺陈背景、勾勒人物、确立风格，更以高度凝练的象征系统，为后续百回巨构埋下严密而深邃的伏笔。


### XML格式化输出

In [2]:
from langchain_core.output_parsers import XMLOutputParser

parser = XMLOutputParser()
query = "请提供《三体》这本书的信息"
# 提供结构模板（不带具体值！）
xml_template = """
<book>
    <title>书名</title>
    <author>作者</author>
    <year>年份</year>
    <genre>类型</genre>
</book>
"""
prompt = f"""
你是一个图书数据库助手。请根据你的知识，以严格的XML格式回答以下请求。

要求：
- 只输出XML，不要任何其他文字、解释或Markdown。
- 使用以下结构（可重复<genre>标签表示多个类型）：
{xml_template}

请求：{query}
"""
response = llm.invoke(prompt)
xml_result = parser.parse(response)
print("解析结果：", xml_result)

解析结果： {'book': [{'title': '三体'}, {'author': '刘慈欣'}, {'year': '2008'}, {'genre': '科幻'}]}


In [4]:
print(response)

<book>
    <title>三体</title>
    <author>刘慈欣</author>
    <year>2008</year>
    <genre>科幻</genre>
</book>


In [ ]:
from langchain_core.output_parsers import XMLOutputParser
from langchain_core.prompts import PromptTemplate


query = "请提供《三体》这本书的信息"

parser = XMLOutputParser()

prompt = """
{format_str}

要求：
- 只输出XML，不要任何其他文字、解释或Markdown

请求：{query}
"""

template = PromptTemplate.from_template(prompt)
template = template.partial(format_str=parser.get_format_instructions())

response = llm.invoke(template.format(query=query))

print("解析结果：", response)

解析结果： <book>
   <title>三体</title>
   <author>刘慈欣</author>
   <publication_year>2008</publication_year>
   <genre>科幻小说</genre>
   <language>中文</language>
   <publisher>重庆出版社</publisher>
   <series>三体系列</series>
   <volume>第一卷</volume>
   <pages>302</pages>
   <isbn>978-7-5366-9293-0</isbn>
</book>


### 列表输出

In [6]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser


parser = CommaSeparatedListOutputParser()

# 构建提示
format_str = parser.get_format_instructions()
text_to_analyze = "人工智能、机器学习、深度学习、神经网络、自然语言处理、计算机视觉"

prompt = f"""
请从以下文本中提取所有的技术术语：
{text_to_analyze}

要求：
{format_str}

注意：只提取技术术语，不要添加解释。
"""

# 调用并解析
response = llm.invoke(prompt)
result_list = parser.parse(response)

print(f"提取到 {len(result_list)} 个术语：")
for i, term in enumerate(result_list, 1):
    print(f"{i}. {term}")



提取到 6 个术语：
1. 人工智能
2. 机器学习
3. 深度学习
4. 神经网络
5. 自然语言处理
6. 计算机视觉


In [8]:
# 直接处理已知的逗号分隔字符串
csv_string = "Python,JavaScript,Java,C++,Go"
parsed_from_string = parser.parse(csv_string)
print(f"\n直接解析CSV字符串：{parsed_from_string}")
csv_string.split(',')


直接解析CSV字符串：['Python', 'JavaScript', 'Java', 'C++', 'Go']


['Python', 'JavaScript', 'Java', 'C++', 'Go']

### 自定义结构化

In [ ]:
from langchain_core.output_parsers import BaseOutputParser

class KeyValueParser(BaseOutputParser):
    def parse(self,text:str) -> dict:
        result = {}    # 存放结果字典
        lines = text.strip().split('\n')   # 数据预处理

        for line in lines:
            if ':' in line:
                key,value = line.split(':',1)
                result[key.strip()] = value.strip()
        return result

    def get_format_instructions(self):
        return '请使用键值对格式输出，每行一个，格式为：key:value'
    
# 创建对象
parser = KeyValueParser()

format_str = parser.get_format_instructions()

prompt = f"""
请提取以下文本中的关键信息：
"姓名：张三，年龄：30，职业：软件工程师，城市：北京"

输出格式：
{format_str}
"""

response = llm.invoke(prompt)
response

'姓名:张三  \n年龄:30  \n职业:软件工程师  \n城市:北京'

In [11]:
data = parser.parse(response)
print('自定义解析器结果:')
for key,value in data.items():
    print(f'{key}:{value}')

自定义解析器结果:
姓名:张三
年龄:30
职业:软件工程师
城市:北京


### 声明式模型

In [5]:
from langchain_community.chat_models import ChatZhipuAI

# 获取环境中KEY
zhipu_key = os.environ.get('ZHIPU_KEY')

# 设置KEY
os.environ["ZHIPUAI_API_KEY"] = zhipu_key

# 创建模型
llm = ChatZhipuAI(model="glm-4.6v-flash")

In [ ]:
from pydantic import BaseModel,Field


class UserProfile(BaseModel):
    name:str =Field(description='用户名')
    age:int = Field(description='年龄')
    is_student:bool = Field(description='是否为在校学生')
    tags:list[str] = Field(description='兴趣标签',default_factory=list)

# 声明式模型
structured_llm = llm.with_structured_output(UserProfile)

# 调用模型
rs = structured_llm.invoke("用户叫李四，25岁，在读研究生，喜欢AI和编程。")
print(rs)

name='李四' age=25 is_student=True tags=['AI', '编程']


In [7]:
rs.model_dump()

{'name': '李四', 'age': 25, 'is_student': True, 'tags': ['AI', '编程']}